In [1]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))


Number of entries: 1100


In [2]:
print("Example_entry:\n",data[50])

Example_entry:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [3]:
def format_input(entry):
    instruction_test=(
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction :\n{entry['instruction']}" 
    )
    input_text=f"\n\n### Input:\n {entry['input']}" if entry['input'] else ""
    return instruction_test+input_text


In [4]:
model_input=format_input(data[50])
desired_response=f"\n\n### Response:\n {data[50]['output']}"
print(model_input+desired_response)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction :
Identify the correct spelling of the following word.

### Input:
 Ocassion

### Response:
 The correct spelling is 'Occasion.'


In [5]:
train_portion=int(len(data)*0.85)
test_portion=int(len(data)*0.1)
val_portion=len(data)-train_portion-test_portion
train_data=data[:train_portion]
test_data=data[train_portion:train_portion+test_portion]
val_data=data[train_portion+test_portion:]
print(f"{len(train_data)}\n{len(val_data)}\n{len(test_data)}")

935
55
110


In [6]:
import tiktoken


In [7]:
import torch
from torch.utils.data import Dataset,DataLoader

class InstructionDataset(Dataset):
    def __init__(self,data,tokenizer):
        self.data=data
        self.encoded_texts=[]

        for entry in data:
            instruction_plus_input=format_input(entry)
            response_text=f"\n\n### Response:\n{entry['output']}"
            full_text=instruction_plus_input+response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)
        


In [8]:
tokenizer=tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>",allowed_special={"<|endoftext|>"}))

[50256]


In [12]:
def custom_collate_draft_1(
        batch,
        pad_token_id=50256,
        device="cpu"
):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list=[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        inputs_list.append(inputs)
    inputs_tensor=torch.stack(inputs_list).to(device)
    return inputs_tensor

In [13]:
inputs_1=[0,1,2,3,4]
inputs_2=[5,6]
inputs_3=[7,8,9]
batch=(
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [17]:
def custom_collate_draft_2(   batch,
        pad_token_id=50256,
        device="cpu"
):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list,targets_list=[],[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        inputs_list.append(inputs)
        targets=torch.tensor(padded[1:])
        targets_list.append(targets)
    inputs_tensor=torch.stack(inputs_list).to(device)
    targets_tensor=torch.stack(targets_list).to(device)
    return inputs_tensor,targets_tensor

In [18]:
def custom_collate_fn(   batch,
        pad_token_id=50256,
        device="cpu",
        ignore_index=-100,
        allowed_max_length=None

):
    batch_max_length=max(len(item)+1 for item in batch)
    inputs_list,targets_list=[],[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(
            new_item+[pad_token_id]*(batch_max_length-len(new_item))
        )
        inputs=torch.tensor(padded[:-1])
        
        targets=torch.tensor(padded[1:])
        mask=targets==pad_token_id
        indices=torch.nonzero(mask).squeeze()
        if indices.numel()>1:
            targets[indices[1:]]=ignore_index
        if allowed_max_length is not None:
            inputs=inputs[:allowed_max_length]
            targets=targets[:allowed_max_length]
        inputs_list.append(inputs)
        targets_list.append(targets)
    inputs_tensor=torch.stack(inputs_list).to(device)
    targets_tensor=torch.stack(targets_list).to(device)
    return inputs_tensor,targets_tensor

In [19]:
inputs_1=[0,1,2,3,4]
inputs_2=[5,6]
inputs_3=[7,8,9]
batch=(
    inputs_1,
    inputs_2,
    inputs_3
)
ipt,trt=custom_collate_fn(batch=batch)
print(ipt)
print(trt)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [21]:
logits=torch.tensor(
    [[-1.0,1.0],
      [-0.5,1.5]]
)
targets_1=torch.tensor([0,1])
loss_1=torch.nn.functional.cross_entropy(logits,targets_1)
print(loss_1)

tensor(1.1269)


In [23]:
logits_2=torch.tensor(
    [[-1.0,1.0],
     [-0.5,1.5],
     [-0.5,1.5]]
)
targets_2=torch.tensor([0,1,1])
loss_2=torch.nn.functional.cross_entropy(logits_2,targets_2)
print(loss_2)

tensor(0.7936)
